<a href="https://colab.research.google.com/github/Santiago-Echeverri-Arteaga/Fisica_Computacional_2/blob/master/curso_2026_2/03_redes_fundamentos/31_activaciones_diseno_y_comparacion.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg"
       alt="Abrir en Colab"/>
</a>

# Funciones de activación: diseño y comparación

**Pregunta guía:** ¿Podemos proponer una activación y evaluarla científicamente?<br>
**Duración sugerida:** 4 horas.<br>
**Entorno:** CPU; datos incluidos o generados en memoria.

El orden de trabajo es siempre: problema → matemática → implementación
mínima → biblioteca → evaluación → interpretación física.


## Qué debe aportar una activación

Una activación introduce no linealidad y controla el flujo de gradientes.
Analizaremos rango, derivada, saturación, suavidad, costo y media de las
salidas. No existe una función universalmente mejor: la comparación debe
mantener arquitectura, datos, inicialización y presupuesto constantes.

Diseñaremos
$$\phi_{onda}(x)=\tanh(x)+0.1\sin(2x),\qquad
\phi'_{onda}(x)=1-\tanh^2(x)+0.2\cos(2x).$$
Es una hipótesis del estudiante, no una mejora asegurada.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEMILLA = 42

def sigmoid(x):
    # Forma estable frente a overflow.
    positiva = x >= 0
    salida = np.empty_like(x, dtype=float)
    salida[positiva] = 1 / (1 + np.exp(-x[positiva]))
    exp_x = np.exp(x[~positiva])
    salida[~positiva] = exp_x / (1 + exp_x)
    return salida

def relu(x): return np.maximum(0.0, x)
def d_relu(x): return (x > 0).astype(float)
def tanh(x): return np.tanh(x)
def d_tanh(x): return 1 - np.tanh(x) ** 2
def swish(x): return x * sigmoid(x)
def d_swish(x):
    s = sigmoid(x)
    return s + x * s * (1 - s)
def onda(x): return np.tanh(x) + 0.1 * np.sin(2 * x)
def d_onda(x): return 1 - np.tanh(x) ** 2 + 0.2 * np.cos(2 * x)

activaciones = {
    "ReLU": (relu, d_relu),
    "tanh": (tanh, d_tanh),
    "swish": (swish, d_swish),
    "onda_propia": (onda, d_onda),
}


In [ ]:
z = np.linspace(-6, 6, 600)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for nombre, (f, df) in activaciones.items():
    axes[0].plot(z, f(z), label=nombre)
    axes[1].plot(z, df(z), label=nombre)
axes[0].set(title="activaciones", xlabel="z", ylabel="$\\phi(z)$")
axes[1].set(title="derivadas", xlabel="z", ylabel="$\\phi'(z)$")
for ax in axes: ax.legend()
plt.show()


## Verificación numérica de la derivada

Antes de entrenar una activación propia comprobamos su derivada con
diferencias centrales. El error debe decrecer hasta que el redondeo de
punto flotante domina.


In [ ]:
puntos = np.linspace(-3, 3, 101)
h = 1e-5
derivada_numérica = (onda(puntos + h) - onda(puntos - h)) / (2 * h)
error = np.max(np.abs(derivada_numérica - d_onda(puntos)))
print(f"Error máximo de la derivada propia: {error:.2e}")
assert error < 1e-7


## Ablación controlada en una red NumPy

Entrenamos una red 2→16→1 sobre las mismas lunas. Reiniciamos exactamente
los pesos para cada activación; sólo cambia $\phi$. La salida usa sigmoid
y entropía cruzada binaria.


In [ ]:
X, y = make_moons(n_samples=900, noise=0.22, random_state=SEMILLA)
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y[:, None], test_size=0.25, stratify=y, random_state=SEMILLA
)
escalador = StandardScaler().fit(X_dev)
X_dev, X_test = escalador.transform(X_dev), escalador.transform(X_test)

def entrenar(activación, derivada, épocas=900, lr=0.05):
    rng = np.random.default_rng(SEMILLA)
    W1 = rng.normal(0, np.sqrt(1 / 2), (2, 16)); b1 = np.zeros((1, 16))
    W2 = rng.normal(0, np.sqrt(1 / 16), (16, 1)); b2 = np.zeros((1, 1))
    historial = []
    n = len(X_dev)
    for época in range(épocas):
        z1 = X_dev @ W1 + b1
        a1 = activación(z1)
        prob = sigmoid(a1 @ W2 + b2)
        dz2 = (prob - y_dev) / n
        dW2 = a1.T @ dz2; db2 = dz2.sum(axis=0, keepdims=True)
        dz1 = (dz2 @ W2.T) * derivada(z1)
        dW1 = X_dev.T @ dz1; db1 = dz1.sum(axis=0, keepdims=True)
        W1 -= lr * dW1; b1 -= lr * db1
        W2 -= lr * dW2; b2 -= lr * db2
        if época % 25 == 0:
            eps = 1e-9
            pérdida = -np.mean(y_dev*np.log(prob+eps)+(1-y_dev)*np.log(1-prob+eps))
            historial.append((época, pérdida))
    prob_test = sigmoid(activación(X_test @ W1 + b1) @ W2 + b2)
    return np.asarray(historial), accuracy_score(y_test, prob_test >= 0.5), log_loss(y_test, prob_test)

filas = []
plt.figure(figsize=(8, 4))
for nombre, (f, df) in activaciones.items():
    hist, acc, pérdida = entrenar(f, df)
    filas.append({"activación": nombre, "accuracy_test": acc, "log_loss_test": pérdida})
    plt.plot(hist[:, 0], hist[:, 1], label=nombre)
plt.yscale("log")
plt.xlabel("época"); plt.ylabel("entropía cruzada train"); plt.legend(); plt.show()
display(pd.DataFrame(filas).set_index("activación"))


Una sola corrida no basta para declarar ganadora. Repita semillas y
reporte media, dispersión, velocidad y estabilidad. Examine también la
distribución de preactivaciones y gradientes por capa.

**Proyecto de activación:** proponga $\phi$ y su derivada; verifíquela
numéricamente; justifique inicialización; compare al menos cinco semillas;
incluya un caso donde falle. Una función más complicada sólo merece
conservarse si ofrece evidencia reproducible.
